# RESUME.ipynb — v9 第二場 session 續跑

`model-train_v9.ipynb` 的 160 epochs 約需 15.8 小時（實測 355 s/epoch），
超過 Kaggle 單場上限（互動式 9 小時 / Save & Run All 12 小時），因此第二場用本檔續跑。

### 執行前確認
1. Session 1 已執行 **Step.6** 壓出 `runs.zip` 並上傳成 Kaggle Dataset。
2. 下方 Step.R 的 `RESUME_SRC` 已改成該 Dataset 的實際路徑。

### 本檔的組成
| 段落 | 來源 | 為什麼需要 |
| --- | --- | --- |
| Step.1 | 與主 notebook 逐字相同 | 重建 `/kaggle/working/data.yaml` 與資料集 |
| Step.2 | 與主 notebook 逐字相同 | `bbox_iou` 必須被換成 Wise-Inner-MPDIoU，否則續跑的 loss 與第一場不同 |
| Step.3 | 與主 notebook 逐字相同 | `last.pt` 內含 pickle 過的自訂類別，沒註冊就載不回來 |
| Step.R | 本檔專屬 | 還原 runs 目錄並 `resume=True` |
| Step.6 | 與主 notebook 逐字相同 | 壓出最終輸出 |

**不需要**跑 Step.4（產生 yaml）、Step.4.5（架構驗證）、Step.5（訓練），
架構與超參數都從 `last.pt` 內的紀錄還原。

> ⚠ Step.1 / 2 / 3 是從 `model-train_v9.ipynb` 複製過來的。
> 若日後修改主 notebook 的這幾段，記得同步回本檔，否則續跑的模型會與第一場不一致。

# Step.1 環境檢查與資料集安全準備

In [ ]:
# 1. 檢查 GPU 狀態
!nvidia-smi
# 2. 安裝與更新 Ultralytics 與 TensorFlow 套件
!pip install -q -U ultralytics tensorflow pyyaml

In [ ]:
import os
import shutil
import sys
import yaml
import ultralytics
print(f"▷ Ultralytics 版本: {ultralytics.__version__}")

In [ ]:
# 1. 定義進度條渲染與資料集複製校驗函式
def render_progress_bar(current, total, task_name="檔案同步複製中", bar_length=25):
    percent = (current / total) * 100 if total > 0 else 100.0
    filled_length = int(bar_length * current // total) if total > 0 else bar_length
    bar = '█' * filled_length + '░' * (bar_length - filled_length)
    
    # 輸出兩行格式：顯示步驟名稱與進度條
    sys.stdout.write(f"\r▷ 正在執行 [{task_name}] | 進度: [{bar}] {percent:5.1f}% ({current}/{total})")
    sys.stdout.flush()
def copy_and_verify_dataset(src_dir, dst_dir):
    if not os.path.exists(src_dir):
        print(f"▷ 錯誤：找不到來源資料集目錄 {src_dir}")
        return False

    # 步驟 1: 收集來源端所有檔案路徑
    src_files = []
    for root, _, files in os.walk(src_dir):
        for file in files:
            rel_path = os.path.relpath(os.path.join(root, file), src_dir)
            src_files.append(rel_path)
    
    total_files = len(src_files)
    print(f"▷ 來源資料集掃描完成，共計 {total_files} 個檔案")

    # 步驟 2: 逐檔複製並動態刷新進度條
    for idx, rel_path in enumerate(src_files, 1):
        src_path = os.path.join(src_dir, rel_path)
        dst_path = os.path.join(dst_dir, rel_path)
        
        os.makedirs(os.path.dirname(dst_path), exist_ok=True)
        shutil.copy2(src_path, dst_path)
        
        # 每 100 筆或最後一筆刷新終端機顯示
        if idx % 100 == 0 or idx == total_files:
            render_progress_bar(idx, total_files, task_name="▷ 檔案同步複製中")
    
    print("\n\n▷ 正在檢查複製檔案")

    # 步驟 3: 複製後檢查機制（比對檔案存在性與 Byte 大小）
    missing_files = []
    corrupted_files = []
    
    dst_files_set = set()
    for root, _, files in os.walk(dst_dir):
        for file in files:
            rel_path = os.path.relpath(os.path.join(root, file), dst_dir)
            dst_files_set.add(rel_path)

    for rel_path in src_files:
        if rel_path not in dst_files_set:
            missing_files.append(rel_path)
        else:
            src_sz = os.path.getsize(os.path.join(src_dir, rel_path))
            dst_sz = os.path.getsize(os.path.join(dst_dir, rel_path))
            if src_sz != dst_sz:
                corrupted_files.append(rel_path)

    # 步驟 4: 輸出校驗結果報告
    print("≡" * 60)
    print("▷ 資料集複製完整性校驗：")
    print(f"  ▶ 來源檔案總數 (Source)     : {len(src_files)}")
    print(f"  ▶ 目標檔案總數 (Destination): {len(dst_files_set)}")
    print(f"  ▶ 遺漏檔案數   (Missing)    : {len(missing_files)}")
    print(f"  ▶ 損毀/大小不符(Corrupted)  : {len(corrupted_files)}")
    
    if not missing_files and not corrupted_files:
        print("▷ 檢查通過")
        print("≡" * 60)
        return True
    else:
        print("▷ 檢查失敗")
        if missing_files:
            print(f"▷ 遺漏檔案: {missing_files[:5]}")
        if corrupted_files:
            print(f"▷ 損毀檔案: {corrupted_files[:5]}")
        print("≡" * 60)
        return False

In [ ]:
# 2. 執行複製與路徑配置
src_dataset_dir = "/kaggle/input/datasets/yentsai9183/datasets-yolo26-v5"
dst_dataset_dir = "/kaggle/working/datasets-yolo26-v5"

# 執行複製與校驗
is_success = copy_and_verify_dataset(src_dataset_dir, dst_dataset_dir)

if is_success:
    # 3. 更新 data.yaml 路徑
    new_yaml_path = "/kaggle/working/data.yaml"
    orig_yaml_path = f"{dst_dataset_dir}/data.yaml"
    
    if os.path.exists(orig_yaml_path):
        with open(orig_yaml_path, 'r', encoding='utf-8') as f:
            yaml_data = yaml.safe_load(f)
        
        yaml_data['path'] = dst_dataset_dir
        yaml_data['train'] = "train/images"
        yaml_data['val'] = "valid/images"
        yaml_data['test'] = "test/images"
        
        with open(new_yaml_path, 'w', encoding='utf-8') as f:
            yaml.safe_dump(yaml_data, f, default_flow_style=False)
        print(f"▷ data.yaml 已更新\n▷ 新設定檔路徑為: {new_yaml_path}")

    # 4. 清理 BOM 標記與舊快取
    print("▷ 正在刪除BOM標籤和Cache")
    modified_count = 0
    deleted_cache_count = 0
    
    for subdir, _, files in os.walk(dst_dataset_dir):
        if "labels" in subdir:
            for file in files:
                if file.endswith('.txt'):
                    file_path = os.path.join(subdir, file)
                    with open(file_path, 'rb') as f:
                        header = f.read(3)
                    if header == b'\xef\xbb\xbf':
                        with open(file_path, 'r', encoding='utf-8-sig') as f:
                            content = f.read()
                        with open(file_path, 'w', encoding='utf-8') as f:
                            f.write(content)
                        modified_count += 1
                        
        for file in files:
            if file.endswith('.cache'):
                os.remove(os.path.join(subdir, file))
                deleted_cache_count += 1
                
    print(f"    ▶ 已自動修正 BOM 檔案數: {modified_count}")
    print(f"    ▶ 已清理舊快取檔案數: {deleted_cache_count}")
    print("▷ Step.1 完成")

# Step.2 加入 Wise-Inner-MPDIoU

In [ ]:
import torch
import math
import ultralytics.utils.loss
import ultralytics.utils.metrics

# ══════════════════════════════════════════════════════════════════════════
# Wise-Inner-MPDIoU
#   Inner-IoU  : 依 ratio 縮放輔助框，放大微小病蟲害的位移梯度
#   MPDIoU     : 左上/右下頂點對角約束，避免定位漂移
#   WIoU v3    : 依離群度動態聚焦，抑制背景反光造成的低品質樣本
#
# ── 相容性要求（照 Ultralytics 原始碼實測）─────────────────────────────
# 1. ultralytics/utils/loss.py 的 BboxLoss.forward 呼叫方式為
#       iou = bbox_iou(pred[fg], target[fg], xywh=False, CIoU=True)
#    → 簽名必須吃得下 GIoU/DIoU/CIoU 三個關鍵字，否則第一個 batch 就 TypeError。
# 2. 同一行後面是
#       loss_iou = ((1.0 - iou) * weight).sum() / target_scores_sum
#    weight 形狀為 [N,1]（target_scores[fg].sum(-1, keepdim=True)）。
#    因此本函式必須回傳 [N,1]；若用 box1[..., 0] 索引會退化成 [N]，
#    相乘會廣播成 [N,N]，損失錯誤且記憶體暴增 → 一律用 chunk(4, -1)。
# 3. 回傳值語意 = 1 - total_loss，如此 (1.0 - iou) 恰好等於 total_loss。
# ══════════════════════════════════════════════════════════════════════════

# WIoU v3 需要 L_IoU 的滑動平均來估計離群度，以動量方式維護
_WIOU_STATE = {"iou_mean": 1.0}


def bbox_wise_inner_mpdiou(
    box1, box2, xywh=True, GIoU=False, DIoU=False, CIoU=False, eps=1e-7,
    ratio=0.7,          # Inner-IoU 輔助框縮放比（<1 放大梯度）
    lambda_mpd=1.0,     # MPDIoU 頂點懲罰權重
    lambda_inner=1.0,   # Inner-IoU 損失權重
    alpha=1.9, delta=3.0, momentum=0.02,   # WIoU v3 聚焦係數超參數
):
    if xywh:
        (cx1, cy1, w1, h1), (cx2, cy2, w2, h2) = box1.chunk(4, -1), box2.chunk(4, -1)
        hw1, hh1, hw2, hh2 = w1 / 2, h1 / 2, w2 / 2, h2 / 2
        b1_x1, b1_x2, b1_y1, b1_y2 = cx1 - hw1, cx1 + hw1, cy1 - hh1, cy1 + hh1
        b2_x1, b2_x2, b2_y1, b2_y2 = cx2 - hw2, cx2 + hw2, cy2 - hh2, cy2 + hh2
    else:
        b1_x1, b1_y1, b1_x2, b1_y2 = box1.chunk(4, -1)
        b2_x1, b2_y1, b2_x2, b2_y2 = box2.chunk(4, -1)
        w1, h1 = (b1_x2 - b1_x1).clamp(0), (b1_y2 - b1_y1).clamp(0)
        w2, h2 = (b2_x2 - b2_x1).clamp(0), (b2_y2 - b2_y1).clamp(0)
        cx1, cy1 = (b1_x1 + b1_x2) / 2, (b1_y1 + b1_y2) / 2
        cx2, cy2 = (b2_x1 + b2_x2) / 2, (b2_y1 + b2_y2) / 2

    # ── 1. 標準 IoU ────────────────────────────────────────────────────
    inter = (b1_x2.minimum(b2_x2) - b1_x1.maximum(b2_x1)).clamp(0) * \
            (b1_y2.minimum(b2_y2) - b1_y1.maximum(b2_y1)).clamp(0)
    union = w1 * h1 + w2 * h2 - inter + eps
    iou = inter / union

    # ── 2. Inner-IoU：兩框各自以中心點依 ratio 縮放後再算 IoU ──────────
    r = ratio / 2
    i1_x1, i1_x2, i1_y1, i1_y2 = cx1 - w1 * r, cx1 + w1 * r, cy1 - h1 * r, cy1 + h1 * r
    i2_x1, i2_x2, i2_y1, i2_y2 = cx2 - w2 * r, cx2 + w2 * r, cy2 - h2 * r, cy2 + h2 * r
    inner_inter = (i1_x2.minimum(i2_x2) - i1_x1.maximum(i2_x1)).clamp(0) * \
                  (i1_y2.minimum(i2_y2) - i1_y1.maximum(i2_y1)).clamp(0)
    inner_union = (w1 * h1 + w2 * h2) * ratio ** 2 - inner_inter + eps
    inner_iou = inner_inter / inner_union

    # ── 3. MPDIoU 頂點對角約束 ─────────────────────────────────────────
    cw = b1_x2.maximum(b2_x2) - b1_x1.minimum(b2_x1)
    ch = b1_y2.maximum(b2_y2) - b1_y1.minimum(b2_y1)
    c_diag_sq = cw.pow(2) + ch.pow(2) + eps
    d1_sq = (b1_x1 - b2_x1).pow(2) + (b1_y1 - b2_y1).pow(2)
    d2_sq = (b1_x2 - b2_x2).pow(2) + (b1_y2 - b2_y2).pow(2)
    mpdiou_penalty = (d1_sq + d2_sq) / c_diag_sq

    # ── 4. WIoU v3 ────────────────────────────────────────────────────
    # R_WIoU 的分母（外接框對角線）必須 detach，否則梯度會傾向「放大外接框」
    # 來降低損失，與收斂方向相反（WIoU 論文 Eq.7 的 * 標記即為此意）。
    l_iou = 1.0 - iou
    center_dist_sq = (cx1 - cx2).pow(2) + (cy1 - cy2).pow(2)
    r_wiou = torch.exp(center_dist_sq / c_diag_sq.detach())

    if torch.is_grad_enabled() and l_iou.numel() > 0:   # 僅在訓練階段更新，避免驗證污染統計量
        _WIOU_STATE["iou_mean"] = (1 - momentum) * _WIOU_STATE["iou_mean"] \
                                  + momentum * l_iou.detach().mean().item()
    beta = l_iou.detach() / max(_WIOU_STATE["iou_mean"], eps)   # 離群度
    focus = beta / (delta * alpha ** (beta - delta))            # beta == delta 時恰為 1
    loss_wiou = focus * r_wiou * l_iou

    total_loss = loss_wiou + lambda_mpd * mpdiou_penalty + lambda_inner * (1.0 - inner_iou)
    return 1.0 - total_loss


# ── 注入 ──────────────────────────────────────────────────────────────
# loss.py 是 `from .metrics import bbox_iou`，覆寫 loss 模組命名空間有效 → 影響訓練梯度。
ultralytics.utils.loss.bbox_iou = bbox_wise_inner_mpdiou
# metrics.py 本身不呼叫 bbox_iou；且 tal.py 的 TaskAlignedAssigner 是在 import 時
# 就把 bbox_iou 綁進自己的命名空間，這行「不會」影響標籤分配器與驗證階段的 mAP。
# 保留此行僅為一致性 —— 驗證指標仍走標準 box_iou，故 v8/v9 的 mAP 可直接對照。
ultralytics.utils.metrics.bbox_iou = bbox_wise_inner_mpdiou

# ── 自我檢查：用與 loss.py 完全相同的呼叫方式驗證簽名與形狀 ──────────
_p = torch.tensor([[10.0, 10.0, 30.0, 30.0], [0.0, 0.0, 10.0, 10.0]], requires_grad=True)
_t = torch.tensor([[12.0, 12.0, 32.0, 32.0], [50.0, 50.0, 60.0, 60.0]])
_iou = ultralytics.utils.loss.bbox_iou(_p, _t, xywh=False, CIoU=True)
assert _iou.shape == (2, 1), f"回傳形狀必須為 [N,1]，實際為 {tuple(_iou.shape)}"
assert torch.isfinite(_iou).all(), "回傳值含 NaN/Inf"
(1.0 - _iou).sum().backward()
assert torch.isfinite(_p.grad).all(), "梯度含 NaN/Inf"
print(f"▷ 簽名/形狀/梯度檢查通過  loss(重疊框)={float((1 - _iou[0]).detach()):.4f}  loss(不相交框)={float((1 - _iou[1]).detach()):.4f}")
print("▷ Step.2 完成")

# Step.3 定義並註冊 StarTripletBlock（ADown 沿用內建版）

In [ ]:
import torch
import torch.nn as nn
from ultralytics.nn.modules.conv import Conv
import ultralytics.nn.tasks
import ultralytics.nn.modules

# ══════════════════════════════════════════════════════════════════════════
# ADown 說明
#   直接沿用 Ultralytics 內建 ADown（YOLOv9 原始實作，已在 parse_model 的
#   base_modules 清單內，可自動取得 c1/c2 並套用 width 縮放）。
#   內建版流程：avg_pool(k=2,s=1) 平滑 → chunk → 一路 stride-2 Conv、
#   另一路 max_pool(k=3,s=2)+1x1 Conv → cat，兩路輸出解析度一致。
#   v9 初版自訂的 ADown 是先整體 avg_pool(s=2) 再對其中一路 max_pool(s=2)，
#   兩路變成 H/2 與 H/4，torch.cat 必定 RuntimeError，故不再自行覆寫。
# ══════════════════════════════════════════════════════════════════════════


class TripletAttention(nn.Module):
    """無降維三元注意力：C-W、H-C、H-W 三個分支的空間/通道交互權重平均。"""

    def __init__(self, k=7):
        super().__init__()
        # act=False：Conv 預設帶 SiLU，接 sigmoid 會變成雙重非線性，注意力會被壓平
        self.cw = Conv(2, 1, k, 1, act=False)
        self.hc = Conv(2, 1, k, 1, act=False)
        self.hw = Conv(2, 1, k, 1, act=False)

    @staticmethod
    def _pool(t):
        return torch.cat([t.max(1, keepdim=True)[0], t.mean(1, keepdim=True)], dim=1)

    def forward(self, x):
        p1 = x.permute(0, 2, 1, 3).contiguous()                       # B,H,C,W → 沿 H 池化
        o1 = (p1 * self.cw(self._pool(p1)).sigmoid()).permute(0, 2, 1, 3).contiguous()

        p2 = x.permute(0, 3, 2, 1).contiguous()                       # B,W,H,C → 沿 W 池化
        o2 = (p2 * self.hc(self._pool(p2)).sigmoid()).permute(0, 3, 2, 1).contiguous()

        o3 = x * self.hw(self._pool(x)).sigmoid()                     # 沿 C 池化（標準空間注意力）
        return (o1 + o2 + o3) / 3.0


class StarBlock(nn.Module):
    """StarNet 星形運算單元（通道數不變）。

    DW → 1x1 雙路升維後逐元素相乘（隱式高維映射）→ TripletAttention → 1x1 降維 → DW → 殘差。
    """

    def __init__(self, c, mlp_ratio=3, k=7):
        super().__init__()
        c_ = int(c * mlp_ratio)
        self.dw1 = Conv(c, c, k, 1, g=c, act=False)
        self.f1 = Conv(c, c_, 1, 1, act=False)
        self.f2 = Conv(c, c_, 1, 1, act=False)
        self.act = nn.ReLU6()
        self.ta = TripletAttention(k)
        self.g = Conv(c_, c, 1, 1, act=False)
        self.dw2 = Conv(c, c, k, 1, g=c, act=False)

    def forward(self, x):
        y = self.dw1(x)
        y = self.act(self.f1(y)) * self.f2(y)   # star operation
        y = self.ta(y)
        return x + self.dw2(self.g(y))


class StarTripletBlock(nn.Module):
    """c1→c2 投影 + n 個 StarBlock；簽名刻意與 C2f 對齊 (c1, c2, n, ...)。

    v9 初版把殘差寫成 `res + dwconv2(v)`，其中 res 為 c1 通道、dwconv2 輸出 c2 通道，
    只要 c1≠c2 就形狀不符（head 第 19 層即為 c1=96 / c2=32），而且
    `Conv(c1, c2, g=c1)` 在 c2 不被 c1 整除時連建構都會失敗。
    這裡改成先用 1x1 投影對齊通道，之後所有 StarBlock 都在 c2 上做等通道殘差。
    """

    def __init__(self, c1, c2, n=1, mlp_ratio=3, k=7):
        super().__init__()
        self.proj = Conv(c1, c2, 1, 1) if c1 != c2 else nn.Identity()
        self.blocks = nn.Sequential(*(StarBlock(c2, mlp_ratio, k) for _ in range(n)))

    def forward(self, x):
        return self.blocks(self.proj(x))


# ══════════════════════════════════════════════════════════════════════════
# 註冊：為什麼是綁到 `C2f` 這個名字
#
# parse_model 內部的 base_modules / repeat_modules 是在「函式執行時」用
# tasks 模組的 globals 動態建立的 frozenset，但成員名單是寫死的 34 個既有名稱。
# 因此：
#   • 覆寫既有名稱（如 ADown）→ globals 被換掉，該 frozenset 也跟著換成新類別，
#     能正常走 `if m in base_modules` 分支拿到 (c1, c2)。
#   • 新增名稱（如 StarTripletBlock）→ 永遠不在名單內，一定落到 else 分支：
#       c2 = ch[f]（輸出通道記成輸入通道）、args 不做 width 縮放、c1 不注入
#     實際呼叫變成 StarTripletBlock(128) → TypeError: missing 'c2'。
#
# 由於無法把新名稱塞進那兩個 frozenset，這裡把 StarTripletBlock 綁到
# 「base_modules 與 repeat_modules 都有、且本專案 yaml 完全沒用到」的 `C2f`。
# 如此可完整取得與 C3k2 相同的待遇：
#   自動注入 c1、c2 依 width 縮放、n 依 depth 縮放並作為第 3 個參數傳入。
# yaml 中出現的 C2f 一律代表 StarTripletBlock；模型摘要表仍會印出真實類名。
# （已確認 tasks.py 中 `C2f` 僅出現於 import 與這兩個 frozenset，無其他副作用。）
# ══════════════════════════════════════════════════════════════════════════
for _cls in (TripletAttention, StarBlock, StarTripletBlock):
    # 讓 checkpoint pickle 時記錄成 ultralytics.nn.tasks.<name> 而非 __main__.<name>
    _cls.__module__ = "ultralytics.nn.tasks"
    setattr(ultralytics.nn.tasks, _cls.__name__, _cls)
    setattr(ultralytics.nn.modules, _cls.__name__, _cls)

ultralytics.nn.tasks.C2f = StarTripletBlock   # ★ 別名注入

# ── 形狀自我檢查（涵蓋 c1==c2 與 c1!=c2 兩種情形）────────────────────
_x = torch.randn(2, 96, 32, 32)
_m = StarTripletBlock(96, 32, n=1)
assert _m(_x).shape == (2, 32, 32, 32), _m(_x).shape
_y = torch.randn(2, 32, 32, 32)
assert StarTripletBlock(32, 32, n=2)(_y).shape == (2, 32, 32, 32)
_a = ultralytics.nn.tasks.ADown(32, 64)
assert _a(torch.randn(2, 32, 32, 32)).shape == (2, 64, 16, 16)
print("▷ StarTripletBlock / ADown 形狀檢查通過")
print("▷ 提醒：日後載入 v9 的 best.pt / last.pt（含匯出 ONNX、TFLite）之前，")
print("        必須先執行 Step.2 與 Step.3，否則 pickle 找不到自訂類別會報錯。")
print("▷ Step.3 完成")

# Step.R 續跑

`resume=True` 會沿用 `last.pt` 內記錄的全部超參數（`epochs=160`、`batch`、`lr`、`close_mosaic`…），
因此下方不要再傳任何訓練參數。

**不要改用 `time=` 參數限制時數** —— Ultralytics 會用實測 epoch 時間反推並改寫 `args.epochs`
（`engine/trainer.py:619`），續跑的總輪數會被改掉。

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Step.R 續跑
# ═══════════════════════════════════════════════════════════════════════
import os
import re
import shutil
import zipfile
import ultralytics.nn.tasks
import ultralytics.utils.loss
from ultralytics import YOLO

# ── 0. 前置檢查 ───────────────────────────────────────────────────────
# Step.3：last.pt 內含 pickle 過的自訂類別，沒註冊就反序列化不回來
for _name in ("StarTripletBlock", "StarBlock", "TripletAttention"):
    assert hasattr(ultralytics.nn.tasks, _name), \
        f"ultralytics.nn.tasks 缺少 {_name} —— 請先執行本檔的 Step.3"
assert ultralytics.nn.tasks.C2f is ultralytics.nn.tasks.StarTripletBlock, \
    "C2f 別名未生效 —— 請先執行本檔的 Step.3"
# Step.2：漏跑不會報錯，但會靜默改用內建 CIoU，續跑的 loss 就與第一場不一致
assert ultralytics.utils.loss.bbox_iou.__name__ == "bbox_wise_inner_mpdiou", \
    "bbox_iou 仍是內建 CIoU —— 請先執行本檔的 Step.2，否則續跑的損失函式與第一場不同"
print("▷ 0/4 自訂 loss 與模組均已就位")

# ── 1. 還原上一場 session 的整個 runs 目錄 ────────────────────────────
# ★ 改成你的 Kaggle Dataset 路徑。可指向 Step.6 壓出的 runs.zip，或解開後的 runs 資料夾。
RESUME_SRC = "/kaggle/input/v9-session1/runs.zip"

RUN_NAME = "YOLO26n_P2_Citrus_MuSGD_v9"
WORK_RUNS = "/kaggle/working/runs"
RUN_DIR = f"{WORK_RUNS}/detect/{RUN_NAME}"
WDIR = f"{RUN_DIR}/weights"

assert os.path.exists(RESUME_SRC), f"找不到 {RESUME_SRC}，請確認 Kaggle Dataset 路徑"
os.makedirs(WORK_RUNS, exist_ok=True)
if RESUME_SRC.endswith(".zip"):
    with zipfile.ZipFile(RESUME_SRC) as z:
        z.extractall(WORK_RUNS)
else:
    shutil.copytree(RESUME_SRC, WORK_RUNS, dirs_exist_ok=True)
assert os.path.isdir(WDIR), (
    f"還原後找不到 {WDIR}。Step.6 的 runs.zip 內層應為 detect/{RUN_NAME}/weights/...，"
    "若結構不同請調整 RESUME_SRC 或解壓目標。")
print(f"▷ 1/4 已還原 runs 目錄：{sorted(os.listdir(WDIR))}")

# ── 2. 挑出可續跑的 checkpoint ────────────────────────────────────────
# final_eval() 會對 last.pt / best.pt 執行 strip_optimizer()，把 epoch 改成 -1
# 並清掉 optimizer/EMA，那種檔案無法續跑。可用的優先序：
#   resume_from.pt（Session 1 的 callback 在 strip 之前留的複本）
#   epochN.pt（save_period 產出，不會被 strip，取 N 最大者）
#   last.pt（只有在 Session 1 是被強制中斷、來不及 strip 時才可用）
import torch

def usable(path):
    """回傳該 ckpt 已完成的輪數；不可續跑則回傳 None。"""
    try:
        ck = torch.load(path, map_location="cpu", weights_only=False)
    except Exception as e:
        print(f"    {os.path.basename(path)}: 讀取失敗 {type(e).__name__}")
        return None
    ep, opt = ck.get("epoch", -1), ck.get("optimizer")
    del ck
    return ep + 1 if (ep is not None and ep >= 0 and opt is not None) else None

cands = [os.path.join(WDIR, "resume_from.pt")]
cands += sorted((os.path.join(WDIR, f) for f in os.listdir(WDIR)
                 if re.fullmatch(r"epoch\d+\.pt", f)),
                key=lambda p: int(re.search(r"\d+", os.path.basename(p)).group()), reverse=True)
cands.append(os.path.join(WDIR, "last.pt"))

CKPT, DONE = None, None
for c in cands:
    if not os.path.exists(c):
        continue
    n = usable(c)
    print(f"    {os.path.basename(c):<18} " + (f"已完成 {n} 輪，可續跑" if n else "已被 strip，不可續跑"))
    if n and CKPT is None:
        CKPT, DONE = c, n

assert CKPT, (
    "沒有任何可續跑的 checkpoint。所有權重的 epoch 都是 -1，代表 Session 1 的訓練迴圈"
    "正常結束並執行了 strip_optimizer()——若那是因為 patience=30 觸發早停，"
    "表示訓練本來就已收斂完成，不需要續跑。")
print(f"▷ 2/4 選用 {os.path.basename(CKPT)}（已完成 {DONE} 輪）")

# ── 3. 續跑 ───────────────────────────────────────────────────────────
# resume=True 會沿用 ckpt 內記錄的全部超參數（epochs=160、batch、lr、close_mosaic…），
# 因此下方不要再傳任何訓練參數。
model = YOLO(CKPT)
print(f"▷ 3/4 開始續跑：第 {DONE + 1} 輪 → 第 160 輪")
results = model.train(resume=True)
print("▷ 4/4 RESUME 訓練完畢")

# Step.6 Output整理

In [ ]:
import os
import shutil

# 定義工作區與 uns目錄路徑
working_dir = "/kaggle/working"
runs_dir = os.path.join(working_dir, "runs")
zip_output_path = os.path.join(working_dir, "runs")

# 檢查runs目錄並壓縮
if os.path.exists(runs_dir):
    print("▷ 正在壓縮訓練輸出")
    shutil.make_archive(zip_output_path, 'zip', runs_dir)
    
    zip_full_path = f"{zip_output_path}.zip"
    if os.path.exists(zip_full_path):
        size_mb = os.path.getsize(zip_full_path) / (1024 * 1024)
        print(f"▷ 壓縮成功 {zip_full_path} ({size_mb:.2f} MB)")
else:
    print(f"▷ 壓縮失敗: 找不到 '{runs_dir}' 資料夾")